# OPM Call/Price Range Optimizer

Reusable end-to-end notebook: given a spreadsheet with P/N, Total Calls, and DN Price columns, it:

1. Searches for **Call Range** breakpoints (A/B/C/D, given LA=2-3, C1=1, C0=0 are fixed) so that, across the full dataset:
   - Count of P/N:  D < C < B < A < LA < C1 < C0   (increasing)
   - Total Calls:   D > C > B > A > LA > C1 > C0   (decreasing)
2. Searches for **DN Price** breakpoints (4 whole-number limits -> 5 bands: to L1, to L2, to L3, to L4, high value) so that:
   - Count of P/N:  band1 > band2 > band3 > band4 > band5  (decreasing)
   - Total Calls:   band1 > band2 > band3 > band4 > band5  (decreasing)
   Items with DN Price <= 0 or blank get their own "Empty" category instead of being dropped, so every P/N is represented.
3. **Jointly optimizes both axes together**: rather than picking Call and Price breakpoints independently, it alternates between the two searches - each time picking whichever valid option (for that axis) pairs best with the other axis's current breakpoints - until neither one changes anymore.
4. Builds a new workbook with the raw data, the breakpoints as editable blue inputs, and two live-formula pivot tables (Count of P/N, Total Calls) cross-tabbed by Price Range (rows, including "Empty") x Call Range (columns). Each pivot includes a **% of Total** column and row so you can see each Call/Price Range's share of the full population, and the Grand Total covers every P/N (including Empty-price items).

Run the cells in order, then edit the **Configuration** cell with your file paths/column names and run the **Execute** cell at the bottom.

In [1]:
import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

FONT = "Arial"
CALL_CATS = ["D", "C", "B", "A", "LA", "C1", "C0"]

## Pivot scoring helpers (used for joint optimization)

These build the actual Price Range x Call Range grid for a given pair of breakpoints, and score how "good" that grid looks (fewer empty cells, bigger smallest cell, less imbalance). The joint optimizer below uses this to judge candidates against each other.

In [2]:
def classify_calls(calls_arr, a_max, b_max, c_max):
    """Vectorized version of the Call Range logic used in the Excel formula."""
    cats = np.empty(len(calls_arr), dtype=object)
    calls_arr = np.asarray(calls_arr)
    cats[calls_arr == 0] = "C0"
    cats[calls_arr == 1] = "C1"
    cats[(calls_arr >= 2) & (calls_arr <= 3)] = "LA"
    cats[(calls_arr >= 4) & (calls_arr <= a_max)] = "A"
    cats[(calls_arr > a_max) & (calls_arr <= b_max)] = "B"
    cats[(calls_arr > b_max) & (calls_arr <= c_max)] = "C"
    cats[calls_arr > c_max] = "D"
    return cats


def classify_price(price_arr, p1, p2, p3, p4):
    """Vectorized version of the Price Range logic used in the Excel formula.
    Items with Price <= 0 (or missing) get their own "Empty" category instead
    of being dropped, so they still show up in the pivot / Grand Total."""
    price_arr = np.asarray(price_arr)
    cats = np.full(len(price_arr), "Empty", dtype=object)
    valid = price_arr > 0
    cats[valid & (price_arr <= p1)] = "to L1"
    cats[valid & (price_arr > p1) & (price_arr <= p2)] = "to L2"
    cats[valid & (price_arr > p2) & (price_arr <= p3)] = "to L3"
    cats[valid & (price_arr > p3) & (price_arr <= p4)] = "to L4"
    cats[valid & (price_arr > p4)] = "high value"
    return cats


PRICE_CATS = ["to L1", "to L2", "to L3", "to L4", "high value", "Empty"]


def build_pivot_counts(df, call_bps, price_bps):
    """Builds the Price Range x Call Range count grid for a given pair of
    breakpoints - the same grid PIVOT 1 in the output workbook shows.
    Includes an "Empty" price row so every P/N is represented."""
    a_max, b_max, c_max = call_bps
    p1, p2, p3, p4 = price_bps
    call_cats = classify_calls(df["Calls"].values, a_max, b_max, c_max)
    price_cats = classify_price(df["Price"].values, p1, p2, p3, p4)
    sub = pd.DataFrame({"CallCat": call_cats, "PriceCat": price_cats})
    pivot = pd.crosstab(sub["PriceCat"], sub["CallCat"])
    pivot = pivot.reindex(index=PRICE_CATS, columns=CALL_CATS, fill_value=0)
    return pivot


def score_pivot(pivot):
    """Lower is better. Primary: fewer empty cells. Secondary: a bigger smallest
    non-empty cell. Tertiary: less relative spread among the non-empty cells.
    The "Empty" price row is excluded from scoring since it isn't something
    either search can influence."""
    scoring_rows = [r for r in PRICE_CATS if r != "Empty"]
    flat = pivot.loc[scoring_rows].values.flatten()
    zero_cells = int((flat == 0).sum())
    nonzero = flat[flat > 0]
    if len(nonzero) == 0:
        return (zero_cells, 0, 0.0)
    min_nonzero = int(nonzero.min())
    mean_nonzero = float(nonzero.mean())
    cv = float(nonzero.std() / mean_nonzero) if mean_nonzero > 0 else 0.0
    return (zero_cells, -min_nonzero, cv)

## Data loading

In [3]:
def load_data(path, sheet=None, pn_col=None, price_col=None, calls_col=None):
    xls = pd.ExcelFile(path)
    sheet = sheet or xls.sheet_names[0]
    raw = pd.read_excel(path, sheet_name=sheet)

    def find_col(explicit, keyword):
        if explicit:
            return explicit
        for c in raw.columns:
            if keyword.lower() in str(c).lower():
                return c
        raise ValueError(f"Could not auto-detect a column matching '{keyword}'. "
                          f"Available columns: {list(raw.columns)}")

    pn_col = find_col(pn_col, "P/N")
    price_col = find_col(price_col, "Price")
    calls_col = find_col(calls_col, "Call")

    df = raw[[pn_col, price_col, calls_col]].copy()
    df.columns = ["PN", "Price", "Calls"]
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce").fillna(0)
    df["Calls"] = pd.to_numeric(df["Calls"], errors="coerce").fillna(0).astype(int)
    return df

## Call Range breakpoint search (A/B/C/D boundaries, given LA/C1/C0 fixed)

In [4]:
def find_call_breakpoints(df, price_bps=None):
    """
    Finds valid A/B/C/D call-range breakpoints.

    If `price_bps` is None, picks the single best breakpoint set using only
    the Calls distribution (the original marginal search).

    If `price_bps` is given, evaluates every VALID breakpoint set (all of
    them still satisfy the base D<C<B<A<LA<C1<C0 count/calls pattern) against
    the resulting Price Range x Call Range pivot grid, and picks whichever
    one produces the best-looking grid together with the given price_bps.
    This is the "call step" of the alternating joint optimization.
    """
    LA_count = int(df["Calls"].isin([2, 3]).sum())
    LA_sum = int(df.loc[df["Calls"].isin([2, 3]), "Calls"].sum())

    sub = df[df["Calls"] >= 4]
    if sub.empty:
        raise ValueError("No items with Calls >= 4; cannot build A/B/C/D bands.")
    calls_arr = sub["Calls"].values
    vals = np.sort(sub["Calls"].unique()).tolist()

    def band_stats(lo, hi):
        mask = (calls_arr >= lo) & (calls_arr <= hi)
        return int(mask.sum()), int(calls_arr[mask].sum())

    results = []
    for ai, a_max in enumerate(vals):
        cntA, sumA = band_stats(4, a_max)
        if cntA >= LA_count or sumA <= LA_sum:
            continue
        for bi in range(ai + 1, len(vals)):
            b_max = vals[bi]
            cntB, sumB = band_stats(a_max + 1, b_max)
            if cntB >= cntA or sumB <= sumA:
                continue
            for ci in range(bi + 1, len(vals)):
                c_max = vals[ci]
                cntC, sumC = band_stats(b_max + 1, c_max)
                if cntC >= cntB or sumC <= sumB:
                    continue
                cntD, sumD = band_stats(c_max + 1, vals[-1])
                if cntD == 0 or cntD >= cntC or sumD <= sumC:
                    continue
                minratio = min(cntB / cntA, cntC / cntB, cntD / cntC, cntA / LA_count)
                results.append((minratio, a_max, b_max, c_max))

    if not results:
        raise ValueError("No valid Call Range breakpoints found for this dataset "
                          "(the A/B/C/D/LA/C1/C0 monotonic pattern isn't achievable).")

    if price_bps is None:
        results.sort(key=lambda x: x[0])
        _, a_max, b_max, c_max = results[0]
        return a_max, b_max, c_max

    scored = []
    for minratio, a_max, b_max, c_max in results:
        pivot = build_pivot_counts(df, (a_max, b_max, c_max), price_bps)
        score = score_pivot(pivot)
        scored.append((score, -minratio, a_max, b_max, c_max))
    scored.sort(key=lambda x: (x[0], x[1]))
    _, _, a_max, b_max, c_max = scored[0]
    return a_max, b_max, c_max

## Price Range breakpoint search (4 limits -> 5 bands)

In [5]:
def find_price_breakpoints(df, price_cap=None, whole_number=True, max_cap_search=200, call_bps=None):
    """
    Finds valid Price Range breakpoints (Limit of range 1-4).

    `price_cap` is now OPTIONAL - Limit of range 1 no longer has to be any
    particular value. If `price_cap` is given, it's used only as a soft
    starting point for the search (to bias it toward that neighborhood);
    if it's None (the default), the search starts near the point that
    covers about 1/5 of the population and works outward from there.

    `whole_number`: if True (default), all 4 breakpoints come out as whole
    numbers.

    If `call_bps` is None, picks the single solution nearest the starting
    point using only the Price/Calls distribution (the original marginal
    search).

    If `call_bps` is given, gathers several distinct valid solutions (each
    starting from a different Limit-of-range-1 candidate), evaluates each
    against the resulting Price Range x Call Range pivot grid together with
    the given call_bps, and picks whichever pairs best. This is the "price
    step" of the alternating joint optimization.
    """
    pdata = df[df["Price"] > 0].copy()
    if pdata.empty:
        raise ValueError("No items with Price > 0; cannot build price bands.")

    if whole_number:
        pdata["Price"] = np.maximum(1, np.round(pdata["Price"].values)).astype(int)

    vals, inv = np.unique(pdata["Price"].values, return_inverse=True)
    cnt_per_val = np.bincount(inv, minlength=len(vals))
    sum_per_val = np.bincount(inv, weights=pdata["Calls"].values, minlength=len(vals))
    cum_cnt = np.cumsum(cnt_per_val)
    cum_sum = np.cumsum(sum_per_val)
    N = len(vals)
    total_cnt = int(cum_cnt[-1])
    total_sum = float(cum_sum[-1])

    def cum_at(i):
        if i < 0:
            return 0, 0.0
        return int(cum_cnt[i]), float(cum_sum[i])

    def band(prev_idx, idx):
        pc, ps = cum_at(prev_idx)
        c, s = cum_at(idx)
        return c - pc, s - ps

    def greedy_max(prev_idx, prev_cnt, prev_sum, min_start, max_idx):
        lo, hi, best = min_start, max_idx, None
        while lo <= hi:
            mid = (lo + hi) // 2
            bc, bs = band(prev_idx, mid)
            if bc < prev_cnt and bs < prev_sum and bc > 0:
                best = mid
                lo = mid + 1
            else:
                hi = mid - 1
        return best

    def dfs_from(idx1, tries_budget=150):
        cnt1, sum1 = cum_at(idx1)
        if cnt1 == 0:
            return None
        max2 = greedy_max(idx1, cnt1, sum1, idx1 + 1, N - 2)
        if max2 is None:
            return None
        idx2, t2 = max2, 0
        while idx2 > idx1 and t2 < tries_budget:
            cnt2, sum2 = band(idx1, idx2)
            if 0 < cnt2 < cnt1 and sum2 < sum1:
                max3 = greedy_max(idx2, cnt2, sum2, idx2 + 1, N - 2)
                if max3 is not None:
                    idx3, t3 = max3, 0
                    while idx3 > idx2 and t3 < tries_budget:
                        cnt3, sum3 = band(idx2, idx3)
                        if 0 < cnt3 < cnt2 and sum3 < sum2:
                            max4 = greedy_max(idx3, cnt3, sum3, idx3 + 1, N - 1)
                            if max4 is not None:
                                idx4, t4 = max4, 0
                                while idx4 > idx3 and t4 < tries_budget:
                                    cnt4, sum4 = band(idx3, idx4)
                                    if 0 < cnt4 < cnt3 and sum4 < sum3:
                                        cnt5 = total_cnt - cum_at(idx4)[0]
                                        sum5 = total_sum - cum_at(idx4)[1]
                                        if 0 < cnt5 < cnt4 and sum5 < sum4:
                                            return (idx1, idx2, idx3, idx4)
                                    idx4 -= 1
                                    t4 += 1
                        idx3 -= 1
                        t3 += 1
            idx2 -= 1
            t2 += 1
        return None

    def finalize(i1, i2, i3, i4):
        if whole_number:
            return int(vals[i1]), int(vals[i2]), int(vals[i3]), int(vals[i4])
        return float(vals[i1]), float(vals[i2]), float(vals[i3]), float(vals[i4])

    target_idx1 = int(np.searchsorted(cum_cnt, max(1, total_cnt // 5), side="left"))
    if price_cap is not None:
        cap_idx_candidates = np.where(vals <= price_cap)[0]
        start_idx = cap_idx_candidates.max() if len(cap_idx_candidates) > 0 else target_idx1
    else:
        start_idx = target_idx1

    if call_bps is None:
        sol = None
        for off in range(0, max_cap_search):
            idx1 = start_idx + off
            if idx1 >= N - 3:
                break
            sol = dfs_from(idx1)
            if sol:
                break
        if not sol:
            raise ValueError("No valid Price Range breakpoints found for this dataset.")
        return finalize(*sol)

    # Joint mode: gather several distinct valid solutions, one per starting
    # point, then score each against the fixed call_bps.
    found = []
    for off in range(0, max_cap_search):
        idx1 = start_idx + off
        if idx1 >= N - 3:
            break
        sol = dfs_from(idx1)
        if sol:
            found.append(sol)

    if not found:
        raise ValueError("No valid Price Range breakpoints found for this dataset.")

    scored = []
    for (i1, i2, i3, i4) in found:
        p1, p2, p3, p4 = finalize(i1, i2, i3, i4)
        pivot = build_pivot_counts(df, call_bps, (p1, p2, p3, p4))
        score = score_pivot(pivot)
        scored.append((score, -i1, p1, p2, p3, p4))
    scored.sort(key=lambda x: (x[0], x[1]))
    _, _, p1, p2, p3, p4 = scored[0]
    return p1, p2, p3, p4

## Joint (alternating) optimization

Instead of picking Call Range and Price Range breakpoints independently, this alternates between the two searches, each time picking the option that pairs best with the other axis's current breakpoints (fewest empty cells in the Price Range x Call Range grid, biggest smallest cell, least imbalance). It repeats until neither breakpoint set changes anymore, or `max_iters` is hit.

In [6]:
def optimize_joint(df, price_cap=None, whole_number=True, max_cap_search=200, max_iters=10, verbose=True):
    """
    Alternating (block coordinate ascent) optimization:
      1. Start from each axis's independent best (marginal) breakpoints.
      2. Re-pick Price breakpoints that pair best with the current Call
         breakpoints.
      3. Re-pick Call breakpoints that pair best with the new Price
         breakpoints.
      4. Repeat until neither changes, or max_iters is reached.

    Returns (call_bps, price_bps, n_iterations).
    """
    call_bps = find_call_breakpoints(df)
    price_bps = find_price_breakpoints(
        df, price_cap=price_cap, whole_number=whole_number, max_cap_search=max_cap_search
    )
    if verbose:
        print(f"Initial (independent) search -> Call={call_bps}, Price={price_bps}")

    for it in range(1, max_iters + 1):
        new_price_bps = find_price_breakpoints(
            df, price_cap=price_cap, whole_number=whole_number,
            max_cap_search=max_cap_search, call_bps=call_bps
        )
        new_call_bps = find_call_breakpoints(df, price_bps=new_price_bps)

        if verbose:
            print(f"Iteration {it} -> Call={new_call_bps}, Price={new_price_bps}")

        if new_call_bps == call_bps and new_price_bps == price_bps:
            call_bps, price_bps = new_call_bps, new_price_bps
            if verbose:
                print(f"Converged after {it} iteration(s).")
            return call_bps, price_bps, it

        call_bps, price_bps = new_call_bps, new_price_bps

    if verbose:
        print(f"Reached max_iters={max_iters} without full convergence; using the latest result.")
    return call_bps, price_bps, max_iters

## Workbook builder

In [7]:
def build_workbook(df, call_bps, price_bps, output_path):
    a_max, b_max, c_max = call_bps
    p1, p2, p3, p4 = price_bps
    price_whole = all(isinstance(v, (int, np.integer)) for v in (p1, p2, p3, p4))
    price_number_format = "#,##0" if price_whole else "#,##0.00"
    n = len(df)
    last_row = 1 + n

    wb = Workbook()
    ws = wb.active
    ws.title = "OPM_Optimized"

    blue = Font(name=FONT, color="0000FF")
    black = Font(name=FONT, color="000000")
    bold = Font(name=FONT, bold=True)
    header_fill = PatternFill("solid", start_color="D9E1F2")
    title_font = Font(name=FONT, bold=True, size=12)
    thin = Side(style="thin", color="BFBFBF")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    headers = ["P/N", "Tot Call", "DN Price", "Call Range", "Price Range"]
    for i, h in enumerate(headers, start=1):
        c = ws.cell(row=1, column=i, value=h)
        c.font = bold
        c.fill = header_fill

    for idx, row in enumerate(df.itertuples(index=False), start=2):
        ws.cell(row=idx, column=1, value=row.PN).font = black
        ws.cell(row=idx, column=2, value=int(row.Calls)).font = black
        ws.cell(row=idx, column=2).number_format = "#,##0"
        ws.cell(row=idx, column=3, value=float(row.Price)).font = black
        ws.cell(row=idx, column=3).number_format = "#,##0.00"
        ws.cell(row=idx, column=4,
                value=f'=IF(B{idx}=0,"C0",IF(B{idx}=1,"C1",IF(B{idx}<=3,"LA",'
                      f'IF(B{idx}<=$H$2,"A",IF(B{idx}<=$H$3,"B",IF(B{idx}<=$H$4,"C","D"))))))').font = black
        ws.cell(row=idx, column=5,
                value=f'=IF(OR(C{idx}="",C{idx}<=0),"Empty",IF(C{idx}<=$K$2,"to "&$K$2,'
                      f'IF(C{idx}<=$K$3,"to "&$K$3,IF(C{idx}<=$K$4,"to "&$K$4,'
                      f'IF(C{idx}<=$K$5,"to "&$K$5,"high value")))))').font = black

    for col, w in zip("ABCDE", (16, 10, 11, 12, 12)):
        ws.column_dimensions[col].width = w

    ws["G1"] = "Call Range Breakpoints (Tot Call)"
    ws["G1"].font = title_font
    ws["G2"] = "A max (Very slow, starts at 4)"; ws["H2"] = a_max
    ws["G3"] = "B max (Slow moving)"; ws["H3"] = b_max
    ws["G4"] = "C max (Medium moving)"; ws["H4"] = c_max
    ws["G5"] = "D = above C max (Fast moving)"
    ws["G6"] = "LA = calls 2-3, C1 = calls 1, C0 = calls 0 (fixed categories)"
    for r in (2, 3, 4):
        ws.cell(row=r, column=8).font = blue
    for r in (2, 3, 4, 5, 6):
        ws.cell(row=r, column=7).font = Font(name=FONT, italic=(r >= 5))
    ws.column_dimensions["G"].width = 34
    ws.column_dimensions["H"].width = 10

    ws["J1"] = "DN Price Range Breakpoints"
    ws["J1"].font = title_font
    ws["J2"] = "Limit of range 1"; ws["K2"] = p1
    ws["J3"] = "Limit of range 2"; ws["K3"] = p2
    ws["J4"] = "Limit of range 3"; ws["K4"] = p3
    ws["J5"] = "Limit of range 4"; ws["K5"] = p4
    ws["J6"] = "Above Limit 4 = high value"
    ws["J7"] = "Items with DN Price <= 0 or blank = Empty"
    for r in (2, 3, 4, 5):
        ws.cell(row=r, column=11).font = blue
        ws.cell(row=r, column=11).number_format = price_number_format
    for r in (2, 3, 4, 5, 6, 7):
        ws.cell(row=r, column=10).font = Font(name=FONT, italic=(r in (6, 7)))
    ws.column_dimensions["J"].width = 22
    ws.column_dimensions["K"].width = 10

    n_empty = int((df["Price"] <= 0).sum())
    if n_empty:
        ws["G8"] = (f"Note: {n_empty} line item(s) have DN Price <= 0 and are classified "
                    f"as \"Empty\" in the Price Range column and both pivot tables below.")
        ws["G8"].font = Font(name=FONT, italic=True, size=9, color="808080")

    data_rng_D = f"$D$2:$D${last_row}"
    data_rng_E = f"$E$2:$E${last_row}"
    data_rng_B = f"$B$2:$B${last_row}"

    def build_pivot(start_row, title, value_formula_maker):
        ws.cell(row=start_row, column=7, value=title).font = title_font
        hdr = start_row + 1
        ws.cell(row=hdr, column=7, value="Price Range \\ Call Range").font = bold
        ws.cell(row=hdr, column=7).fill = header_fill
        for j, cat in enumerate(CALL_CATS, start=8):
            c = ws.cell(row=hdr, column=j, value=cat)
            c.font = bold; c.fill = header_fill; c.alignment = Alignment(horizontal="center")
        gt_col = 8 + len(CALL_CATS)
        c = ws.cell(row=hdr, column=gt_col, value="Grand Total")
        c.font = bold; c.fill = header_fill
        pct_col = gt_col + 1
        c = ws.cell(row=hdr, column=pct_col, value="% of Total")
        c.font = bold; c.fill = header_fill

        # "Empty" is included as its own row so every P/N is represented and
        # Grand Total covers the full population.
        price_labels = [f'="to "&$K$2', f'="to "&$K$3', f'="to "&$K$4', f'="to "&$K$5',
                         "high value", "Empty"]
        first_data_row = hdr + 1
        for i, lbl in enumerate(price_labels):
            r = first_data_row + i
            ws.cell(row=r, column=7, value=lbl).font = black
            for j, cat in enumerate(CALL_CATS, start=8):
                col_letter = get_column_letter(j)
                formula = value_formula_maker(col_letter, f"$G{r}", hdr)
                ws.cell(row=r, column=j, value=formula).font = black
                ws.cell(row=r, column=j).number_format = "#,##0"
            first_cat_col = get_column_letter(8)
            last_cat_col = get_column_letter(7 + len(CALL_CATS))
            ws.cell(row=r, column=gt_col, value=f"=SUM({first_cat_col}{r}:{last_cat_col}{r})").font = bold
            ws.cell(row=r, column=gt_col).number_format = "#,##0"

        gt_row = first_data_row + len(price_labels)
        last_data_row = first_data_row + len(price_labels) - 1
        gt_col_letter = get_column_letter(gt_col)
        overall_total_ref = f"${gt_col_letter}${gt_row}"

        ws.cell(row=gt_row, column=7, value="Grand Total").font = bold
        for j in range(8, gt_col + 1):
            col_letter = get_column_letter(j)
            ws.cell(row=gt_row, column=j,
                    value=f"=SUM({col_letter}{first_data_row}:{col_letter}{last_data_row})").font = bold
            ws.cell(row=gt_row, column=j).number_format = "#,##0"

        # % of Total column: each price row's Grand Total as a % of the overall total.
        for r in range(first_data_row, gt_row):
            ws.cell(row=r, column=pct_col,
                    value=f"={gt_col_letter}{r}/{overall_total_ref}").font = black
            ws.cell(row=r, column=pct_col).number_format = "0.0%"
        ws.cell(row=gt_row, column=pct_col, value=1).font = bold
        ws.cell(row=gt_row, column=pct_col).number_format = "0.0%"

        # % of Total row: each Call Range column's Grand Total as a % of the overall total.
        pct_row = gt_row + 1
        ws.cell(row=pct_row, column=7, value="% of Total").font = bold
        for j in range(8, gt_col + 1):
            col_letter = get_column_letter(j)
            ws.cell(row=pct_row, column=j, value=f"={col_letter}{gt_row}/{overall_total_ref}").font = bold
            ws.cell(row=pct_row, column=j).number_format = "0.0%"
        ws.cell(row=pct_row, column=pct_col, value=1).font = bold
        ws.cell(row=pct_row, column=pct_col).number_format = "0.0%"

        for r in range(hdr, pct_row + 1):
            for j in range(7, pct_col + 1):
                ws.cell(row=r, column=j).border = border
        return pct_row

    def make_count_formula(col_letter, price_ref, hdr_row):
        return f"=COUNTIFS({data_rng_D},{col_letter}${hdr_row},{data_rng_E},{price_ref})"

    def make_sum_formula(col_letter, price_ref, hdr_row):
        return f"=SUMIFS({data_rng_B},{data_rng_D},{col_letter}${hdr_row},{data_rng_E},{price_ref})"

    end1 = build_pivot(9, "PIVOT 1 : Count of P/N", make_count_formula)
    start2 = end1 + 3
    build_pivot(start2, "PIVOT 2 : Total Calls", make_sum_formula)

    for col in ["H", "I", "J", "K", "L", "M", "N", "O", "P"]:
        ws.column_dimensions[col].width = 12

    ws.freeze_panes = "A2"
    wb.save(output_path)
    print(f"Workbook written to {output_path}")

## Configuration

Edit the values below to match your input file, then run this cell and the **Execute** cell that follows.

- `INPUT_FILE`: path to your input .xlsx
- `OUTPUT_FILE`: path to write the optimized workbook
- `SHEET`: sheet name, or `None` to use the first sheet
- `PN_COL`, `PRICE_COL`, `CALLS_COL`: explicit column names, or `None` to auto-detect (partial, case-insensitive match on "P/N", "Price", "Call")
- `PRICE_CAP`: optional. Limit of range 1 no longer has to equal this - it's just a soft starting point for the search. Set to `None` to let the search start from its own natural starting point instead.
- `WHOLE_NUMBER_PRICE`: set to `True` (default) to force all 4 price breakpoints to whole numbers
- `MAX_ITERS`: max number of alternating rounds between the Call and Price searches before giving up on full convergence

In [8]:
INPUT_FILE = "Agc 23 Call Price.xlsx"
OUTPUT_FILE = "OUTPUT.xlsx"
SHEET = "Sheet2"        # or None for first sheet
PN_COL = "P/N"          # or None to auto-detect
PRICE_COL = "DN Price"  # or None to auto-detect
CALLS_COL = "TotCall"  # or None to auto-detect
PRICE_CAP = None            # optional soft starting point; no longer a hard requirement
WHOLE_NUMBER_PRICE = True    # set False to allow decimal price breakpoints
MAX_ITERS = 10000

## Execute

In [9]:
df = load_data(INPUT_FILE, SHEET, PN_COL, PRICE_COL, CALLS_COL)
print(f"Loaded {len(df)} rows.")

call_bps, price_bps, n_iters = optimize_joint(
    df, price_cap=PRICE_CAP, whole_number=WHOLE_NUMBER_PRICE, max_iters=MAX_ITERS
)

print(f"\nFinal Call Range breakpoints -> A max={call_bps[0]}, B max={call_bps[1]}, C max={call_bps[2]}")
print(f"Final Price Range breakpoints -> {price_bps[0]}, {price_bps[1]}, {price_bps[2]}, {price_bps[3]}")

build_workbook(df, call_bps, price_bps, OUTPUT_FILE)

Loaded 1542 rows.
Initial (independent) search -> Call=(7, 25, 99), Price=(11, 22, 40, 93)
Iteration 1 -> Call=(7, 22, 63), Price=(11, 22, 40, 93)
Iteration 2 -> Call=(7, 22, 63), Price=(11, 22, 40, 93)
Converged after 2 iteration(s).

Final Call Range breakpoints -> A max=7, B max=22, C max=63
Final Price Range breakpoints -> 11, 22, 40, 93
Workbook written to OUTPUT.xlsx
